In [1]:
# This notebook is adapted for local machine use (originally for Google Colab)
# Google Colab drive mounting is not needed - using local paths instead
import os
os.chdir('/Users/ron/Desktop/Deeplearning/3.9_lab_yolo_object_detection')
print(f"Working directory: {os.getcwd()}")

Working directory: /Users/ron/Desktop/Deeplearning/3.9_lab_yolo_object_detection


check the dataset

In [2]:
import os

# Check if images and labels directories already exist
base_dir = os.getcwd()
images_dir = os.path.join(base_dir, 'images')
labels_dir = os.path.join(base_dir, 'labels')

if os.path.exists(images_dir) and os.path.exists(labels_dir):
    print(f"Dataset already exists in {base_dir}")
    print(f"Images: {os.listdir(images_dir)[:5]}...")  # Show first 5
    print(f"Labels: {os.listdir(labels_dir)[:5]}...")  # Show first 5
else:
    print("Warning: 'images' or 'labels' directory not found. Please ensure the dataset is in the current directory.")

Dataset already exists in /Users/ron/Desktop/Deeplearning/3.9_lab_yolo_object_detection
Images: ['blue_bottle_blue_bottle_19659.jpg', 'phone_phone_1842297.jpg', 'red_cup_red_cup_3589425.jpg', 'phone_phone_410324.jpg', 'phone_a phone_388387.jpg']...
Labels: ['red_cup_red_cup_2603438.txt', 'blue_bottle_a blue water bottle_774466.txt', 'red_cup_red_cup_2786036.txt', 'phone_phone_1283938.txt', 'blue_bottle_blue_bottle_2408620.txt']...


Divide the training set and validation set

In [3]:
import os
import random
import shutil
from sklearn.model_selection import train_test_split

# Setup path - using current working directory
base_dir = os.getcwd()
images_dir = os.path.join(base_dir, 'images')
labels_dir = os.path.join(base_dir, 'labels')

# Create training and validation folders 创建训练和验证文件夹
train_images_dir = os.path.join(base_dir, 'train', 'images')
val_images_dir = os.path.join(base_dir, 'val', 'images')
train_labels_dir = os.path.join(base_dir, 'train', 'labels')
val_labels_dir = os.path.join(base_dir, 'val', 'labels')

for d in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
    os.makedirs(d, exist_ok=True)

# Get all image files 获取所有图片文件
image_files = [f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
print(f"Found {len(image_files)} images to split")

# Divide the training set and validation set 划分训练集80%和验证集20%
train_files, val_files = train_test_split(image_files, test_size=0.2, random_state=42)

def copy_files(file_list, source_img_dir, source_label_dir, target_img_dir, target_label_dir):
    for f in file_list:
        # copy images
        src_img = os.path.join(source_img_dir, f)
        dst_img = os.path.join(target_img_dir, f)
        shutil.copy(src_img, dst_img)

        # Copy the corresponding label file (assuming the image and label file names are the same, only the suffixes are different) 复制对应的标签文件 (假设图片和标签文件名相同，只是后缀不同)
        label_file = os.path.splitext(f)[0] + '.txt'
        src_label = os.path.join(source_label_dir, label_file)
        dst_label = os.path.join(target_label_dir, label_file)
        if os.path.exists(src_label):
            shutil.copy(src_label, dst_label)
        else:
            print(f"Warning: Label file not found {src_label}")

# Perform copying 执行复制
copy_files(train_files, images_dir, labels_dir, train_images_dir, train_labels_dir)
copy_files(val_files, images_dir, labels_dir, val_images_dir, val_labels_dir)

print(f"Number of training set images: {len(os.listdir(train_images_dir))}")
print(f"Number of validation set images: {len(os.listdir(val_images_dir))}")

Found 135 images to split
Number of training set images: 130
Number of validation set images: 49


Create a dataset configuration file (data.yaml) 创建数据集配置文件 (data.yaml)

In [4]:
import yaml
import os

base_dir = os.getcwd()

data_yaml_content = f"""# Paths of training and validation images 训练和验证图片的路径
path: {base_dir}  # 数据集根目录
train: train/images  # 训练图片路径 (相对于 path)
val: val/images      # 验证图片路径 (相对于 path)

# Number of categories 类别数量
nc: 3

# Category names 类别名称
names: ['red_cup', 'blue_bottle', 'phone']
"""

yaml_path = os.path.join(base_dir, 'data.yaml')
with open(yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f"data.yaml created successfully at {yaml_path}")

data.yaml created successfully at /Users/ron/Desktop/Deeplearning/3.9_lab_yolo_object_detection/data.yaml


 Train YOLO Model 训练 YOLO 模型


In [5]:
#1 Install Ultralytics YOLO (if not already installed)
import subprocess
import sys

try:
    from ultralytics import YOLO
    print("Ultralytics YOLO is already installed")
except ImportError:
    print("Installing ultralytics...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics"])

Ultralytics YOLO is already installed


In [6]:
#2. Start training the YOLO model - the most important part and most time-consuming part

from ultralytics import YOLO
import os
import numpy as np  
import torch

# Verify PyTorch and numpy are available
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")

base_dir = os.getcwd()
data_yaml_path = os.path.join(base_dir, 'data.yaml')

# Check if training data exists
if not os.path.exists(data_yaml_path):
    raise FileNotFoundError(f"data.yaml not found at {data_yaml_path}")

# Load the pre-trained model (using YOLOv8n, where 'n' stands for nano, which is the smallest and fastest version, suitable for my dataset and first attempt)加载预训练模型 (使用YOLOv8n，'n'代表nano，是最小最快的版本，适合我的数据集和首次尝试)
model = YOLO('yolov8n.pt')

# Start training 开始训练
results = model.train(
    data=data_yaml_path,  # Configuration file path配置文件路径
    epochs=10,                 # Training epochs: 30-50 epochs are sufficient for 150 images训练轮次，30轮对于150张图足够了
    imgsz=416,                 # Enter the image size, YOLOv8 defaults to 416-640 输入图片大小，YOLOv8默认640
    batch=8,                   # Batch size, adjusted according to system resources; 8-16 is fine. 批次大小，根据系统资源调整，16没问题
    patience=10,               # Early Stopping，如果10轮验证集指标没提升就停止
    device='cpu',              # Using CPU instead of GPU 使用CPU而不是GPU
    workers=2,                 # Number of data loading processes 数据加载进程数
    verbose=True               # Print detailed information 打印详细信息
)

print("\n✓ Training completed successfully!")

PyTorch version: 2.2.2
NumPy version: 1.26.4
Ultralytics 8.4.21 🚀 Python-3.9.23 torch-2.2.2 CPU (Intel Core i5-8259U 2.30GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/ron/Desktop/Deeplearning/3.9_lab_yolo_object_detection/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train6, nbs=64, nms=False, opse

Evaluate and Test 评估和测试

In [7]:
# Load the trained best model 加载训练好的最佳模型
from ultralytics import YOLO
import os

base_dir = os.getcwd()
best_model_path = os.path.join(base_dir, 'runs', 'detect', 'train6', 'weights', 'best.pt')

if os.path.exists(best_model_path):
    best_model = YOLO(best_model_path)
    data_yaml_path = os.path.join(base_dir, 'data.yaml')
    
    # Evaluate on the validation set 在验证集上评估
    metrics = best_model.val(data=data_yaml_path)
    print(metrics)
else:
    print(f"Model not found at {best_model_path}. Please train the model first.")

Ultralytics 8.4.21 🚀 Python-3.9.23 torch-2.2.2 CPU (Intel Core i5-8259U 2.30GHz)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 129.4±22.4 MB/s, size: 64.9 KB)
val: Scanning /Users/ron/Desktop/Deeplearning/3.9_lab_yolo_object_detection/val/labels.cache... 49 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 49/49 8.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.1s/it 4.6s2.1ss
                   all         49         65      0.863      0.786      0.864      0.519
               red_cup         15         22      0.849      0.818      0.821       0.52
           blue_bottle         22         22      0.806      0.682      0.852      0.484
                 phone         12         21      0.933      0.857      0.919      0.555
Speed: 3.2ms preprocess, 66.3ms inference, 0.0ms loss, 3.2ms postprocess per image
Results saved to 

Detection image from validation set

In [8]:
import cv2
from ultralytics import YOLO
import os
import matplotlib.pyplot as plt
from IPython.display import clear_output

base_dir = os.getcwd()
best_model_path = os.path.join(base_dir, 'runs', 'detect', 'train6', 'weights', 'best.pt')

# Load your trained model
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    
    # Example: Run detection on a sample image from validation set
    val_images_dir = os.path.join(base_dir, 'val', 'images')
    if os.path.exists(val_images_dir):
        image_files = [f for f in os.listdir(val_images_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
        
        if image_files:
            # Test on first image
            test_image_path = os.path.join(val_images_dir, image_files[0])
            print(f"Testing on image: {test_image_path}")
            
            results = model(test_image_path)
            annotated = results[0].plot()
            
            # Display result using matplotlib
            plt.figure(figsize=(12, 8))
            plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
            plt.title("YOLO Detection Result")
            plt.axis('off')
            plt.tight_layout()
            plt.show()
        else:
            print("No images found in validation directory")
    else:
        print(f"Validation images directory not found at {val_images_dir}")
else:
    print(f"Model not found at {best_model_path}. Please train the model first.")
    

Testing on image: /Users/ron/Desktop/Deeplearning/3.9_lab_yolo_object_detection/val/images/phone_phone_410324.jpg

image 1/1 /Users/ron/Desktop/Deeplearning/3.9_lab_yolo_object_detection/val/images/phone_phone_410324.jpg: 288x416 1 blue_bottle, 82.7ms
Speed: 3.0ms preprocess, 82.7ms inference, 6.7ms postprocess per image at shape (1, 3, 288, 416)


<Figure size 1200x800 with 1 Axes>

 Real-time Demo 实时演示

In [9]:
import cv2
from ultralytics import YOLO
import os
import time
import numpy as np
from collections import deque

base_dir = os.getcwd()
best_model_path = os.path.join(base_dir, 'runs', 'detect', 'train4', 'weights', 'best.pt')

def run_detector_on_video(model, video_source=0, output_path=None, confidence_threshold=0.5, max_frames=None):
    """
    Run YOLO detector on webcam or video file with real-time predictions.
    
    Args:
        model: YOLO model object
        video_source: 0 for webcam, or path to video file (str)
        output_path: Path to save output video (optional)
        confidence_threshold: Minimum confidence for detections (0.0 to 1.0)
        max_frames: Maximum frames to process (None for all)
    """
    cap = cv2.VideoCapture(video_source)
    
    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS)) if isinstance(video_source, str) else 30
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Initialize video writer if output path is provided
    video_writer = None
    if output_path:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        video_writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # Performance metrics
    frame_times = deque(maxlen=30)  # Store last 30 frame times
    frame_count = 0
    detection_count = 0
    
    source_type = "Webcam" if video_source == 0 else f"Video: {os.path.basename(video_source)}"
    print(f"Starting detection on {source_type}")
    print(f"Resolution: {width}x{height} @ {fps}fps")
    print("Press 'q' to stop\n")
    
    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            
            # Run detection (YOLO's single-pass architecture)
            start_time = time.time()
            results = model(frame, conf=confidence_threshold, verbose=False)
            inference_time = time.time() - start_time
            frame_times.append(inference_time)
            
            # Visualize results
            annotated_frame = results[0].plot()
            
            # Calculate metrics
            detections = len(results[0].boxes)
            detection_count += detections
            
            # Calculate FPS
            avg_time = np.mean(frame_times)
            fps_display = 1 / avg_time if avg_time > 0 else 0
            
            # Add performance info to frame
            info_text = f"FPS: {fps_display:.1f} | Detections: {detections} | Time: {inference_time*1000:.1f}ms"
            cv2.putText(annotated_frame, info_text, (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            
            # Display frame
            cv2.imshow('YOLO Real-Time Detection', annotated_frame)
            
            # Write to output video if specified
            if video_writer:
                video_writer.write(annotated_frame)
            
            frame_count += 1
            
            # Print progress
            if frame_count % 30 == 0:
                print(f"Processed {frame_count} frames | Avg inference time: {avg_time*1000:.1f}ms | Avg FPS: {fps_display:.1f}")
            
            # Check for max frames
            if max_frames and frame_count >= max_frames:
                break
            
            # Press 'q' to exit
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
                
    finally:
        cap.release()
        if video_writer:
            video_writer.release()
        cv2.destroyAllWindows()
    
    # Print summary
    print(f"Detection Complete Summary:")
    print(f"Total frames processed: {frame_count}")
    print(f"Total detections: {detection_count}")
    print(f"Avg detections per frame: {detection_count/frame_count:.2f}")
    print(f"Avg inference time per frame: {np.mean(frame_times)*1000:.1f}ms")
    print(f"Average FPS: {1/np.mean(frame_times):.1f}")
    print(f"\nYOLO's Architecture Advantage:")
    print(f"- Single-pass detection (original YOLO innovation, ~{inference_time*1000:.1f}ms per frame)")
    print(f"- Introduced by Joseph Redmon et al. for real-time object detection")
    print(f"- Processes entire image in one neural network evaluation")
    if output_path:
        print(f"\nOutput video saved to: {output_path}")


# Check if model exists
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    
    # Example 1: Run on webcam (uncomment to use)
    # Uncomment the line below to run on my webcam (COMMENTED OUT - uncomment to enable)
    # To quit: Press 'q' in the window
    # run_detector_on_video(model, video_source=0, confidence_threshold=0.5)
    
    # Example 2: Run on a video file 
    # Uncomment and modify the path below to run on a video file
    # video_path = os.path.join(base_dir, 'test_video.mp4')
    # output_path = os.path.join(base_dir, 'detection_output.mp4')
    # run_detector_on_video(model, video_source=video_path, output_path=output_path, confidence_threshold=0.5)
    
    print("Real-time detection function is ready!")
    print("\nTo run detection on my webcam, uncomment and execute:")
    print("  run_detector_on_video(model, video_source=0, confidence_threshold=0.5)")
    print("\nTo run detection on a video file, uncomment and execute:")
    print("  run_detector_on_video(model, video_source='path/to/video.mp4', output_path='output.mp4')")
    
else:
    print(f"Model not found at {best_model_path}")
    print("Please train the model first.")

Real-time detection function is ready!

To run detection on my webcam, uncomment and execute:
  run_detector_on_video(model, video_source=0, confidence_threshold=0.5)

To run detection on a video file, uncomment and execute:
  run_detector_on_video(model, video_source='path/to/video.mp4', output_path='output.mp4')


In [10]:
# Testing optional

# ============================================================================
# OPTION 1: Run on Webcam (Real-time detection from your camera)
# ============================================================================
# Uncomment the line below to run detection on your webcam
# Press 'q' to stop the detection

# run_detector_on_video(model, video_source=0, confidence_threshold=0.5)


# ============================================================================
# OPTION 2: Run on a Video File
# ============================================================================
# First, specify my video file path, then uncomment to run

# video_file = '/path/to/my/video.mp4'  # Change this to my video path
# output_video = os.path.join(base_dir, 'detection_output.mp4')
# run_detector_on_video(model, video_source=video_file, output_path=output_video, confidence_threshold=0.5)


# ============================================================================
# OPTION 3: Run on a Single Image ( test one image)
# ============================================================================


def detect_on_image(model, image_path, confidence_threshold=0.5):
    """Run detection on a single image and display result."""
    import matplotlib.pyplot as plt
    
    image = cv2.imread(image_path)
    if image is None:
        print(f"Error: Could not load image from {image_path}")
        return
    
    print(f"Running detection on: {image_path}")
    
    # Run detection
    results = model(image, conf=confidence_threshold)
    annotated = results[0].plot()
    
    # Display
    plt.figure(figsize=(14, 8))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title(f"YOLO Detection - {len(results[0].boxes)} objects detected")
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    # Print detection details
    print(f"\nDetections found: {len(results[0].boxes)}")
    for i, box in enumerate(results[0].boxes):
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])
        class_name = model.names[class_id]
        print(f"  {i+1}. {class_name}: {confidence:.2%} confidence")

# To use OPTION 3, uncomment and modify the line below:
image_path = os.path.join(base_dir, 'test_image.jpg')
detect_on_image(model, image_path, confidence_threshold=0.5)  # Uncomment this line to run


print("QUICKSTART GUIDE")
print("\n📹 OPTION 1 - Webcam Detection (Real-time):")
print("   Uncomment and run: run_detector_on_video(model, video_source=0)")
print("\n📽️  OPTION 2 - Video File Detection:")
print("   1. Set your video path: video_file = '/path/to/video.mp4'")
print("   2. Uncomment and run: run_detector_on_video(model, video_source=video_file)")
print("\n🖼️  OPTION 3 - Single Image Detection:")
print("   1. Set test image path: image_path = '/path/to/image.jpg'")
print("   2. Uncomment and run: detect_on_image(model, image_path)")
print("\n💡 KEY FEATURES:")
print("   ✓ Real-time FPS calculation")
print("   ✓ Per-frame inference time display")
print("   ✓ Detection count and confidence scores")
print("   ✓ Video output saving (Option 2)")
print("   ✓ YOLO's single-pass detection architecture")

Running detection on: /Users/ron/Desktop/Deeplearning/3.9_lab_yolo_object_detection/test_image.jpg

0: 320x416 1 phone, 82.1ms
Speed: 2.9ms preprocess, 82.1ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 416)


<Figure size 1400x800 with 1 Axes>


Detections found: 1
  1. phone: 78.21% confidence
QUICKSTART GUIDE

📹 OPTION 1 - Webcam Detection (Real-time):
   Uncomment and run: run_detector_on_video(model, video_source=0)

📽️  OPTION 2 - Video File Detection:
   1. Set your video path: video_file = '/path/to/video.mp4'
   2. Uncomment and run: run_detector_on_video(model, video_source=video_file)

🖼️  OPTION 3 - Single Image Detection:
   1. Set test image path: image_path = '/path/to/image.jpg'
   2. Uncomment and run: detect_on_image(model, image_path)

💡 KEY FEATURES:
   ✓ Real-time FPS calculation
   ✓ Per-frame inference time display
   ✓ Detection count and confidence scores
   ✓ Video output saving (Option 2)
   ✓ YOLO's single-pass detection architecture
